<font size=10>**NETWORK**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. First Network](#3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
data = pd.read_csv('../data/preprocessed_data.csv')
# data.head()
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 112103 entries, 0 to 112102
Data columns (total 19 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   idcontrato                   112103 non-null  int64  
 1   tipoContrato                 112103 non-null  str    
 2   tipoFimContrato              17039 non-null   str    
 3   CPV                          112103 non-null  str    
 4   adjudicante                  112103 non-null  str    
 5   adjudicatarios               112103 non-null  str    
 6   concorrentes                 84059 non-null   str    
 7   precoBaseProcedimento        112103 non-null  float64
 8   precoContratual              112103 non-null  float64
 9   PrecoTotalEfetivo            112103 non-null  float64
 10  dataDecisaoAdjudicacao       112103 non-null  str    
 11  dataCelebracaoContrato       112103 non-null  str    
 12  dataPublicacao               112103 non-null  str    
 13  dataFechoC

In [5]:
data["dataPublicacao"] = pd.to_datetime(data["dataPublicacao"], errors='coerce')
data["dataCelebracaoContrato"] = pd.to_datetime(data["dataCelebracaoContrato"], errors='coerce')
data["dataDecisaoAdjudicacao"] = pd.to_datetime(data["dataDecisaoAdjudicacao"], errors='coerce')
data["dataFechoContrato"] = pd.to_datetime(data["dataFechoContrato"], errors='coerce')

# <font color='#BFD72F' size=6>**3. First Network**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

In [6]:
data = data.head(3000)

In [7]:
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add edges from adjudicante to adjudicatarios with contract count as weight
for idx, row in data.iterrows():
    adjudicante = row['adjudicante']
    adjudicatarios = row['adjudicatarios']
    preco = row['precoContratual']
    
    # Handle missing values
    if pd.isna(adjudicante) or pd.isna(adjudicatarios):
        continue
    
    # Add edge with weight (contract amount)
    if G.has_edge(adjudicante, adjudicatarios):
        G[adjudicante][adjudicatarios]['weight'] += preco
        G[adjudicante][adjudicatarios]['contracts'] += 1
    else:
        G.add_edge(adjudicante, adjudicatarios, weight=preco, contracts=1)

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")

Number of nodes: 2735
Number of edges: 2692


In [8]:
output_path = '../graphs/gephi_graph01.gexf'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
nx.write_gexf(G, output_path)
print(f'Graph exported to {output_path}')

Graph exported to ../graphs/gephi_graph01.gexf


In [9]:
# import scipy
# import matplotlib.pyplot as plt

# plt.figure(figsize=(14, 10))
# try:
#     pos = nx.spring_layout(G, k=0.15, iterations=20)
# except ImportError:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
#     importlib.invalidate_caches()

# nx.draw(
#     G,
#     pos,
#     with_labels=True,
#     node_size=500,
#     node_color="lightblue",
#     edge_color="gray",
#     font_size=8,
#     arrows=True
# )

# plt.title("Network Graph")
# plt.show()